# Lab 4 · Lập hồ sơ một quận bằng pandas

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành · bài 4**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook demo bài 4 khám phá toàn thành phố. Trong lab này, bạn lập **hồ sơ chi tiết
của một quận** — Ñuñoa — và so sánh quận này với mặt bằng chung của Santiago.

## Cách làm việc trong buổi lab

- Bài tập được chia thành từng bước; mỗi bước có ô `TODO` và phần kiểm tra `assert`.
  Hoàn thành toàn bộ `assert` nghĩa là kết quả đáp ứng yêu cầu.
- Phần khởi động và bài có hướng dẫn: bạn nên **tự gõ, không dùng AI** — các bài kiểm tra
  định kỳ 🚫 đóng ở giờ lý thuyết đánh giá các kỹ năng này.
- Bài tự làm ở cuối được gắn nhãn ✅ mở: bạn được dùng AI, kèm trách nhiệm khai báo
  theo chính sách AI của môn.
- Nếu chưa giải quyết được một bước sau 3 phút, hãy gọi giảng viên thực hành đến hỗ trợ.

## Mục tiêu

Sau buổi lab, bạn sẽ:

1. Chạy thói quen 5 bước và **rút ra số liệu** từ output, không chỉ thực thi lệnh.
2. Lọc bool một và nhiều điều kiện; chọn cột bằng `[[...]]` và `nsmallest`.
3. Tính tỷ lệ bằng mean-trên-bool và thấy NaN ảnh hưởng kết quả thế nào.
4. Xuất một bảng hồ sơ ra file CSV.

## Phần 0 · Khởi động (~10 phút)

In [3]:
import pandas as pd

URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/visualisations/listings.csv")
df = pd.read_csv(URL)
df.shape
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18534 entries, 0 to 18533
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              18534 non-null  int64  
 1   name                            18534 non-null  object 
 2   host_id                         18534 non-null  int64  
 3   host_profile_id                 18534 non-null  int64  
 4   host_name                       18534 non-null  object 
 5   neighbourhood_group             0 non-null      float64
 6   neighbourhood                   18534 non-null  object 
 7   latitude                        18534 non-null  float64
 8   longitude                       18534 non-null  float64
 9   room_type                       18534 non-null  object 
 10  price                           17688 non-null  float64
 11  minimum_nights                  18528 non-null  float64
 12  number_of_reviews               

,id,name,host_id,host_profile_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
0,978070332077815549,luminosa mansarda con balcón,118157228,1468207531264648757,Patricia,NaN,Ñuñoa,-33.437650,-70.583300,Private room,45647.0,4.0,2,2023-12-15,0.06,5,269,0,NaN
1,1069858035768058539,Cómoda habitación bien ubicada con baño privado,462786436,1470084981980135796,Luís Alfonso,NaN,Recoleta,-33.422420,-70.641160,Private room,19856.0,1.0,8,2026-05-03,1.06,1,239,8,NaN
2,37181369,Cómodo departamento Cerca de Metro,279796881,1469472399879355357,Daniel,NaN,Recoleta,-33.423250,-70.645650,Entire home/apt,46776.0,1.0,126,2026-06-25,1.54,1,254,24,NaN
3,1257358306712011993,Acogedor Dormitorio + baño priv.,149097696,1468458667367579708,Francisco,NaN,Recoleta,-33.422683,-70.639869,Private room,25572.0,1.0,0,NaN,NaN,1,365,0,NaN
4,868747298775934782,Departamento frente a est. metro,386871617,1469796602780866978,Patricia,NaN,Recoleta,-33.422690,-70.643690,Entire home/apt,107044.0,2.0,0,NaN,NaN,1,268,0,NaN


In [4]:
# W1 — chọn cột: một tên -> Series, danh sách -> DataFrame
# TODO: lấy cột price thành Series, và bảng con gồm 2 cột name, price
cot_gia = df["price"]
bang_con = df[["name","price"]]

# --- Ô kiểm tra ---
assert type(cot_gia).__name__ == "Series"
assert type(bang_con).__name__ == "DataFrame" and list(bang_con.columns) == ["name", "price"]
print("W1 ổn — nhớ 2 lớp ngoặc cho DataFrame.")

W1 ổn — nhớ 2 lớp ngoặc cho DataFrame.


In [5]:
# W2 — mean trên bool trả về tỷ lệ (quy tắc bài 3, áp dụng trên pandas)
# TODO: tỷ lệ phòng loại "Private room" trong toàn bảng
ty_le_private = (df["room_type"] == "Private room").mean()

# --- Ô kiểm tra ---
assert round(ty_le_private, 3) == 0.184
print(f"W2 ổn: {ty_le_private:.1%} là phòng riêng")

W2 ổn: 18.4% là phòng riêng


## Phần 1 · Hồ sơ quận Ñuñoa (~55 phút)

Tình huống: một nhà đầu tư quan tâm **quận Ñuñoa** và cần bản hồ sơ một trang:
quy mô, mức giá so với thành phố, cơ cấu phòng và mức thiếu dữ liệu của quận.

### Bước 1 · Thói quen 5 bước — chạy và đọc kết quả

Chạy 5 lệnh dưới, rồi **điền các con số đọc được** vào ô kiểm tra bên dưới
(sao chép giá trị từ output, không ước đoán).

In [6]:
df.shape

(18534, 19)

In [7]:
df.sample(3, random_state=1)[["name", "neighbourhood", "room_type", "price"]]

,name,neighbourhood,room_type,price
4714,Depto Las Condes-Estadio Español,Las Condes,Entire home/apt,85588.0
2621,Habitación Matrimonial + cama 1 plaza,Providencia,Private room,30331.0
8659,"Centro De Santiago, Lastrarías",Santiago,Entire home/apt,46218.0


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18534 entries, 0 to 18533
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              18534 non-null  int64  
 1   name                            18534 non-null  object 
 2   host_id                         18534 non-null  int64  
 3   host_profile_id                 18534 non-null  int64  
 4   host_name                       18534 non-null  object 
 5   neighbourhood_group             0 non-null      float64
 6   neighbourhood                   18534 non-null  object 
 7   latitude                        18534 non-null  float64
 8   longitude                       18534 non-null  float64
 9   room_type                       18534 non-null  object 
 10  price                           17688 non-null  float64
 11  minimum_nights                  18528 non-null  float64
 12  number_of_reviews               

In [9]:
df["price"].describe().round(0)

,price
count,17688.0
mean,118200.0
std,1226089.0
min,979.0
25%,41500.0
50%,59000.0
75%,90814.0
max,97000045.0


In [10]:
df["room_type"].value_counts()

,count
room_type,
Entire home/apt,15004
Private room,3413
Shared room,61
Hotel room,56


In [11]:
# TODO: điền 4 con số vừa đọc được từ 5 output trên
so_dong = 18534            # từ shape
so_cot = 19             # từ shape
so_thieu_price = 846     # từ info(): 18534 - (số non-null của price)
so_loai_phong = 4      # từ value_counts(): có mấy loại phòng?

# --- Ô kiểm tra ---
assert so_dong == 18534 and so_cot == 19
assert so_thieu_price == 846 and so_loai_phong == 4
print("Đọc output chuẩn — kỹ năng bị xem nhẹ nhất và cần nhất.")

Đọc output chuẩn — kỹ năng bị xem nhẹ nhất và cần nhất.


### Bước 2 · Cắt riêng quận Ñuñoa

In [12]:
# TODO: lọc các dòng có neighbourhood == "Ñuñoa" (dùng đúng tên xuất hiện trong output)
nu = df[df["neighbourhood"] == "Ñuñoa"]
# TODO: Ñuñoa chiếm bao nhiêu phần của thành phố?
ty_trong = (df["neighbourhood"] == "Ñuñoa").mean()

# --- Ô kiểm tra ---
assert len(nu) == 1813 and round(ty_trong, 3) == 0.098
print(f"Ñuñoa: {len(nu):,} phòng = {ty_trong:.1%} thị trường.")

Ñuñoa: 1,813 phòng = 9.8% thị trường.


### Bước 3 · Lọc kết hợp trong quận

Nhà đầu tư hỏi: *"trong Ñuñoa, có bao nhiêu căn **nguyên căn** giá **từ 60.000 CLP trở
xuống**?"* — phân khúc họ muốn cạnh tranh.

In [13]:
# TODO: đếm số dòng của nu thoả cả hai điều kiện (nhớ ngoặc quanh từng vế + &)
so_can = ((nu["price"] <= 60_000) & (nu["room_type"] == "Entire home/apt")).sum()

# --- Ô kiểm tra ---
assert so_can == 539
print(f"{so_can} căn — phân khúc khá đông.")

539 căn — phân khúc khá đông.


### Bước 4 · Kiểm tra 5 phòng có giá thấp nhất quận

In [14]:
# TODO: dùng nsmallest lấy 5 dòng giá thấp nhất của nu, chỉ giữ 2 cột name và price
re_nhat = nu.nsmallest(5,"price")[["name","price"]]

# --- Ô kiểm tra ---
assert re_nhat.shape == (5, 2) and re_nhat["price"].min() == 2228.0
re_nhat

,name,price
4827,Dormitorio + baño privado ÑUÑOA,2228.0
16514,Confortables habitaciones en Barrio Italia.,3958.0
16525,Moderno apartamento equipado cercano a Metro/Mall,5714.0
4821,"Gran Departamento AMOBLADO, Ñuñoa, metro Grecia",6073.0
4231,Nice and cosy bedroom,6501.0


2.228 CLP/đêm (~60 nghìn đồng) cho một "dormitorio" là mức giá cần được kiểm tra.
Hãy ghi nhận trường hợp này; bài 10 sẽ trình bày quy tắc gắn cờ các giá trị tương tự.

### Bước 5 · So sánh Ñuñoa với thành phố và xem ảnh hưởng của NaN

So từng phòng của Ñuñoa với **trung vị toàn thành phố** (59.000 CLP).

In [15]:
med_tp = df["price"].median()

# TODO: tính tỷ lệ phòng Ñuñoa có giá THẤP HƠN med_tp, theo 2 cách:
ty_le_tho = (nu["price"] < med_tp).mean()        # cách 1: mean trực tiếp trên (nu["price"] < med_tp)
ty_le_sach = (nu["price"].dropna() < med_tp).mean()       # cách 2: như trên nhưng bỏ NaN trước bằng .dropna()

# --- Ô kiểm tra ---
assert round(ty_le_tho, 3) == 0.444
assert round(ty_le_sach, 3) == 0.467
print(f"Thô: {ty_le_tho:.1%} — Sạch: {ty_le_sach:.1%}. Hai con số, lệch 2.3 điểm %!")

Thô: 44.4% — Sạch: 46.7%. Hai con số, lệch 2.3 điểm %!


Vì sao lệch? Phép so `NaN < 59000` trả `False`, nên 90 phòng không khai giá bị đếm
như thể "không rẻ hơn", làm mẫu số tăng thêm 90. Cách 2 loại các dòng này khỏi phép tính.
Bạn cần chọn cách tính phù hợp và nói rõ rằng tỷ lệ được tính trên các phòng có giá. Đây là
bài học `if g is not None` của lab 2, phiên bản pandas.

### Bước 6 · Cơ cấu phòng và mức thiếu — hoàn thiện hồ sơ

In [16]:
# TODO: cơ cấu loại phòng của Ñuñoa theo TỶ LỆ (value_counts với normalize=True)
co_cau_nu = nu["room_type"].value_counts(normalize=True)
# TODO: tỷ lệ thiếu giá của Ñuñoa (%, làm tròn 2 chữ số)
thieu_nu = round((nu["price"].isna().mean() * 100),2)

# --- Ô kiểm tra ---
assert round(co_cau_nu["Entire home/apt"], 3) == 0.804
assert thieu_nu == 4.96
print(f"Nguyên căn {co_cau_nu['Entire home/apt']:.1%} · thiếu giá {thieu_nu}% (toàn TP: 4.56%)")

Nguyên căn 80.4% · thiếu giá 4.96% (toàn TP: 4.56%)


### Bước 7 · Xuất hồ sơ ra file

In [17]:
ho_so = pd.DataFrame([{
    "quan": "Ñuñoa",
    "so_phong": len(nu),
    "ty_trong_thi_truong": round(ty_trong, 3),
    "gia_trung_vi": nu["price"].median(),
    "ty_le_duoi_tv_thanh_pho": round(ty_le_sach, 3),
    "ty_le_thieu_gia_pct": thieu_nu,
}])
ho_so.to_csv("ho_so_nunoa.csv", index=False)

# TODO: đọc lại file vừa ghi để kiểm tra
doc_lai = pd.read_csv("ho_so_nunoa.csv")

# --- Ô kiểm tra ---
assert doc_lai.shape == (1, 6) and doc_lai.loc[0, "so_phong"] == 1813
assert doc_lai.loc[0, "gia_trung_vi"] == 60981.0
print("Hồ sơ đã lưu — trung vị Ñuñoa 60.981 CLP, nhỉnh hơn thành phố 3%.")

Hồ sơ đã lưu — trung vị Ñuñoa 60.981 CLP, nhỉnh hơn thành phố 3%.


## Phần 2 · Bài tự làm ✅ mở (làm sớm tại lớp hoặc làm tại nhà)

Bạn được dùng AI theo quy trình 5 bước; hãy ghi lại prompt chính và cách kiểm chứng.

### Tự làm 1 · Hồ sơ quận thứ hai

Lặp lại Bước 2–7 cho một quận bạn tự chọn (gợi ý: Providencia hoặc Estación Central) và
viết 3 câu so sánh nó với Ñuñoa. Viết code sao cho **chỉ cần đổi một biến `TEN_QUAN`**
là chạy được cho quận bất kỳ; biến này đóng vai trò cấu hình.

### Tự làm 2 · Quận có giá thấp và nhiều lựa chọn

Dùng `df.groupby("neighbourhood")["price"]` với `.agg(["median", "size"])` (xem trước
bài 5): tìm quận có **≥ 300 phòng** và giá trung vị **thấp nhất**. Kiểm chứng lại con số
bằng cách lọc thủ công quận đó và gọi `.median()`.

In [18]:
# tự làm 1

TEN_QUAN = "Providencia"
# Bước 2 · Cắt riêng quận Providencia
quan = df[df["neighbourhood"] == TEN_QUAN]

# Bước 3 · Lọc kết hợp trong quận


# Bước 4 · Kiểm tra 5 phòng có giá thấp nhất quận
# Bước 5 · So sánh Providencia với thành phố và xem ảnh hưởng của NaN
# Bước 6 · Cơ cấu phòng và mức thiếu — hoàn thiện hồ sơ
# Bước 7 · Xuất hồ sơ ra file

In [28]:
# tự làm 2
tk = df.groupby("neighbourhood")["price"].agg(["median","size"])
tk.loc[tk[tk["size"] >= 300]["median"].idxmin()]



,Estación Central
median,39941.0
size,440.0


## Tóm tắt buổi lab

| Nội dung chính | Sẽ gặp lại ở |
|---|---|
| Thói quen 5 bước + đọc số liệu từ output | kiểm tra dataset mới và code do AI tạo |
| Lọc bool & nhiều điều kiện, `nsmallest` | phân tích các phân khúc dữ liệu |
| NaN làm lệch mean-trên-bool (0.444 vs 0.467) | làm sạch dữ liệu (bài 10) |
| Hồ sơ quận → CSV, code đổi 1 biến chạy quận khác | xây dựng phân tích có thể tái sử dụng |

Buổi lý thuyết tiếp theo: index, `groupby` và ghép bảng để tổng hợp nhiều quận.